In [10]:
!pip install rank-bm25 sentence-transformers groq google-generativeai -q

In [11]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]   = userdata.get("GROQ_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [12]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import google.generativeai as genai
from groq import Groq

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

print(" All imports successful")

 All imports successful


In [13]:
CORPUS = [
    # Attention & Transformers
    "The attention mechanism allows a model to weigh the importance of different input tokens when producing each output token.",
    "Transformers use self-attention to compute contextual embeddings by comparing every token against every other token in the sequence.",
    "Multi-head attention runs several attention operations in parallel, letting the model capture different types of relationships simultaneously.",

    # Neural Network Training
    "Gradient descent updates model weights by moving in the direction that minimizes the loss function.",
    "Backpropagation computes gradients layer-by-layer using the chain rule, enabling efficient training of deep networks.",
    "Mini-batch stochastic gradient descent (SGD) averages gradients over a small subset of data, balancing speed and stability.",

    # Optimizers
    "Adam optimizer combines momentum and RMSProp, maintaining per-parameter adaptive learning rates using first and second moment estimates.",

    # Embeddings
    "Word2Vec trains shallow neural networks to produce dense vector representations where semantically similar words cluster together.",
    "Sentence-BERT (SBERT) fine-tunes BERT using siamese networks to generate semantically meaningful sentence embeddings efficiently.",

    # Regularization
    "Dropout randomly zeroes out neurons during training to prevent co-adaptation and reduce overfitting in deep networks.",
    "L2 regularization adds a penalty proportional to the square of the weights to discourage large parameter values.",

    # Retrieval-Augmented Generation
    "Retrieval-Augmented Generation (RAG) enhances LLM responses by fetching relevant documents at inference time instead of relying solely on parametric memory.",
]

print(f"Corpus size: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i:02d}] {doc[:80]}...")

Corpus size: 12 documents
  [00] The attention mechanism allows a model to weigh the importance of different inpu...
  [01] Transformers use self-attention to compute contextual embeddings by comparing ev...
  [02] Multi-head attention runs several attention operations in parallel, letting the ...
  [03] Gradient descent updates model weights by moving in the direction that minimizes...
  [04] Backpropagation computes gradients layer-by-layer using the chain rule, enabling...
  [05] Mini-batch stochastic gradient descent (SGD) averages gradients over a small sub...
  [06] Adam optimizer combines momentum and RMSProp, maintaining per-parameter adaptive...
  [07] Word2Vec trains shallow neural networks to produce dense vector representations ...
  [08] Sentence-BERT (SBERT) fine-tunes BERT using siamese networks to generate semanti...
  [09] Dropout randomly zeroes out neurons during training to prevent co-adaptation and...
  [10] L2 regularization adds a penalty proportional to the squa

In [14]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        """
        Args:
            corpus: list of document strings
            k: RRF constant (default 60, per the original paper)
        """
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        print("Loading SBERT model...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print(" HybridRetriever initialized")

    def _bm25_ranked(self, query: str) -> list[tuple[int, float]]:
        """Returns [(doc_id, bm25_score), ...] sorted by score descending."""
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return ranked

    def _sbert_ranked(self, query: str) -> list[tuple[int, float]]:
        """Returns [(doc_id, cosine_similarity), ...] sorted descending."""
        q_emb = self.sbert.encode([query], convert_to_numpy=True)
        corpus_norm = self.corpus_embeddings / (
            np.linalg.norm(self.corpus_embeddings, axis=1, keepdims=True) + 1e-9
        )
        q_norm = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
        sims = (corpus_norm @ q_norm.T).squeeze()
        ranked = sorted(enumerate(sims.tolist()), key=lambda x: x[1], reverse=True)
        return ranked

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_ranked  = self._bm25_ranked(query)
        sbert_ranked = self._sbert_ranked(query)

        bm25_rank  = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(bm25_ranked)}
        sbert_rank = {doc_id: rank + 1 for rank, (doc_id, _) in enumerate(sbert_ranked)}

        all_ids = set(range(len(self.corpus)))
        rrf_scores = {}
        for doc_id in all_ids:
            r_bm25  = bm25_rank.get(doc_id, len(self.corpus))
            r_sbert = sbert_rank.get(doc_id, len(self.corpus))
            rrf_scores[doc_id] = (
                1 / (self.k + r_bm25) +
                1 / (self.k + r_sbert)
            )

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        results = []
        for doc_id, score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results

In [15]:
print("Loading cross-encoder model...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print(" Cross-encoder loaded")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-rank candidate documents using a cross-encoder.

    Args:
        query:      The ORIGINAL user query (not HyDE-expanded).
        candidates: List of dicts from HybridRetriever.retrieve()
        top_k:      Number of top documents to return

    Returns:
        List of dicts with added "ce_score" field, sorted by ce_score descending.

    Note: Cross-encoder scores can be negative — higher (less negative) = more relevant.
    """
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)  # returns numpy array

    for cand, score in zip(candidates, scores):
        cand["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]

Loading cross-encoder model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Cross-encoder loaded


In [16]:
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

def hyde_expand(user_query: str) -> str:
    """
    Generate a hypothetical document that would answer the query.
    Using temperature=0.0 for deterministic, factual output.
    The hypothetical doc is used as the retrieval query — not shown to the user.
    """
    prompt = (
        "You are an expert in AI and machine learning. "
        "Write a concise, technical 2–3 sentence passage that directly answers "
        "the following question. Write as if it were an excerpt from a textbook.\n\n"
        f"Question: {user_query}\n\nPassage:"
    )
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(temperature=0.0)
    )
    return response.text.strip()

In [18]:

def advanced_rag(user_query: str, verbose: bool = True) -> str:
    if verbose: print(f"\n{'='*60}\n Query: {user_query}")
    hypothetical_doc = hyde_expand(user_query)
    if verbose: print(f"\n HyDE doc:\n   {hypothetical_doc[:120]}...")

    candidates = retriever.retrieve(hypothetical_doc, top_k=6)
    if verbose:
        print(f"\n Hybrid retrieval top-3 (of 6 candidates):")
        for c in candidates[:3]:
            print(f"   RRF={c['rrf_score']} | {c['text'][:75]}")

    top_docs = rerank(user_query, candidates, top_k=3)
    if verbose:
        print(f"\n  After re-ranking:")
        for i, d in enumerate(top_docs):
            print(f"   #{i+1} CE={d['ce_score']:>7.4f} | {d['text'][:75]}")

    context = "\n".join(f"[{i+1}] {d['text']}" for i, d in enumerate(top_docs))
    prompt = (
        f"You are a helpful university AI assistant. "
        f"Answer the student's question using only the provided context. "
        f"Be concise and technical.\n\n"
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        f"Answer:"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=300,
    )
    answer = response.choices[0].message.content.strip()
    if verbose: print(f"\nAnswer:\n{answer}\n{'='*60}")
    return answer


_ = advanced_rag("what is attention?")


 Query: what is attention?

 HyDE doc:
   Attention is a mechanism within neural networks that allows the model to dynamically weigh the importance of different i...

 Hybrid retrieval top-3 (of 6 candidates):
   RRF=0.032787 | The attention mechanism allows a model to weigh the importance of different
   RRF=0.032258 | Multi-head attention runs several attention operations in parallel, letting
   RRF=0.030366 | Gradient descent updates model weights by moving in the direction that mini

  After re-ranking:
   #1 CE= 3.2900 | The attention mechanism allows a model to weigh the importance of different
   #2 CE=-0.9787 | Multi-head attention runs several attention operations in parallel, letting
   #3 CE=-4.2670 | Transformers use self-attention to compute contextual embeddings by compari

Answer:
Attention is a mechanism that allows a model to weigh the importance of different input tokens when producing each output token.


In [19]:
naive_sbert = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embs = naive_sbert.encode(CORPUS, convert_to_numpy=True)

def naive_rag(user_query: str) -> dict:
    """
    Baseline: SBERT cosine retrieval only — no query expansion, no re-ranking.
    Returns top document dict for comparison.
    """
    q_emb = naive_sbert.encode([user_query], convert_to_numpy=True)
    corpus_norm = corpus_embs / (np.linalg.norm(corpus_embs, axis=1, keepdims=True) + 1e-9)
    q_norm = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-9)
    sims = (corpus_norm @ q_norm.T).squeeze()
    top_id = int(np.argmax(sims))
    return {"doc_id": top_id, "score": round(float(sims[top_id]), 4), "text": CORPUS[top_id]}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
TEST_QUERIES = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what makes RAG better than standard LLMs?",
]

comparison_rows = []

for query in TEST_QUERIES:
    # Naïve RAG — top doc
    naive_top = naive_rag(query)

    # Advanced RAG — capture top doc without verbose output
    hyp_doc   = hyde_expand(query)
    candidates = retriever.retrieve(hyp_doc, top_k=6)
    adv_top   = rerank(query, candidates, top_k=1)[0]

    different = " Yes" if naive_top["doc_id"] != adv_top["doc_id"] else " No"

    comparison_rows.append({
        "query":        query,
        "naive_top":    f"[{naive_top['doc_id']}] {naive_top['text'][:60]}...",
        "advanced_top": f"[{adv_top['doc_id']}]  {adv_top['text'][:60]}...",
        "different":    different,
    })
    print(f"✔ Done: '{query[:40]}'")

# Pretty-print results
print("\n" + "="*80)
print("COMPARISON TABLE")
print("="*80)
for row in comparison_rows:
    print(f"\nQuery   : {row['query']}")
    print(f"Naïve   : {row['naive_top']}")
    print(f"Advanced: {row['advanced_top']}")
    print(f"Different? {row['different']}")

✔ Done: 'how do transformers encode meaning?'
✔ Done: 'optimization techniques for training'
✔ Done: 'what makes RAG better than standard LLMs'

COMPARISON TABLE

Query   : how do transformers encode meaning?
Naïve   : [1] Transformers use self-attention to compute contextual embedd...
Advanced: [1]  Transformers use self-attention to compute contextual embedd...
Different?  No

Query   : optimization techniques for training
Naïve   : [3] Gradient descent updates model weights by moving in the dire...
Advanced: [4]  Backpropagation computes gradients layer-by-layer using the ...
Different?  Yes

Query   : what makes RAG better than standard LLMs?
Naïve   : [11] Retrieval-Augmented Generation (RAG) enhances LLM responses ...
Advanced: [11]  Retrieval-Augmented Generation (RAG) enhances LLM responses ...
Different?  No


## Part 6 — Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|---|---|---|
| "how do transformers encode meaning?" | [1] Transformers use self-attention to compute contextual embeddings... | [1] Transformers use self-attention to compute contextual embeddings... | No |
| "optimization techniques for training" | [3] Gradient descent updates model weights by moving in the direction... | [4] Backpropagation computes gradients layer-by-layer using the chain rule... | Yes |
| "what makes RAG better than standard LLMs?" | [11] Retrieval-Augmented Generation (RAG) enhances LLM responses... | [11] Retrieval-Augmented Generation (RAG) enhances LLM responses... | No |

**Observations:**

- **Query 1 & 3:** Both pipelines return the same top doc — the query is clear enough that SBERT alone gets it right.
- **Query 2:** Advanced RAG picks backpropagation over gradient descent, likely because HyDE introduced terms like *chain rule* that shifted retrieval toward a more specific result.
- **Takeaway:** Advanced RAG helps most when the query vocabulary doesn't directly match the corpus.